In [0]:
%run ./libs/mlflow_logger


In [0]:
%run ./libs/model_evaluator

In [0]:
%run ./libs/model_trainer

In [0]:
%run ./libs/report_generator

In [0]:
%run ./libs/model_visualizer

In [0]:
import os
print(os.getcwd())  # current workspace path of this notebook


/Workspace/MOMO-BIGDG/MLOps_NBs/Chstomer_Churn_predictor


In [0]:
%run ./data_generator

In [0]:


class ChurnPredictionPipeline:
    """
    Main orchestrator for the entire ML pipeline.
    Modular, reusable, and easy to configure.
    """
    
    def __init__(self, catalog, schema,viz_path, algorithm='logistic_regression', **algo_params):
        self.spark = SparkSession.builder.appName("ChurnPipeline").getOrCreate()
        self.catalog = catalog
        self.schema = schema
        self.algorithm = algorithm
        self.algo_params = algo_params
        self.viz_path= viz_path
        
        # Feature columns
        self.feature_cols = ['recency', 'frequency', 'monetary', 'tenure_days']
        
        # Components
        self.data_generator = None
        self.trainer = None
        self.evaluator = None
        self.visualizer = None
        self.mlflow_logger = None
        self.report_generator = None
        
    def run(self, n_customers=10000, test_size=0.2, generate_viz=True, 
            save_viz=True, log_to_mlflow=True, export_results=True):
        """
        Run the complete pipeline
        
        Args:
            n_customers: Number of customers to generate
            test_size: Test set proportion
            generate_viz: Generate visualizations
            save_viz: Save visualizations to disk
            log_to_mlflow: Log to MLflow
            export_results: Export to multiple formats
        """
        
        print("\n" + "="*80)
        print("🚀 STARTING CHURN PREDICTION PIPELINE")
        print("="*80)
        
        # ====================================================================
        # STEP 1: Generate Data
        # ====================================================================
        print("\n📊 STEP 1: Generating Data...")
        self.data_generator = ChurnDataGenerator(
            self.spark, self.catalog, self.schema, 
            n_customers=n_customers
        )
        df = self.data_generator.create_feature_table()
        self.data_generator.create_actuals_table()
        
        # ====================================================================
        # STEP 2: Train Model
        # ====================================================================
        print("\n🤖 STEP 2: Training Model...")
        self.trainer = MLModelTrainer(
            feature_cols=self.feature_cols,
            test_size=test_size
        )
        self.trainer.prepare_data(df)
        model = self.trainer.train(algorithm=self.algorithm, **self.algo_params)
        predictions = self.trainer.predict()
        feature_importance = self.trainer.get_feature_importance()
        
        # ====================================================================
        # STEP 3: Evaluate Model
        # ====================================================================
        print("\n📈 STEP 3: Evaluating Model...")
        self.evaluator = ModelEvaluator(predictions)
        metrics = self.evaluator.calculate_all_metrics()
        self.evaluator.print_summary()
        
        # ====================================================================
        # STEP 4: Generate Visualizations
        # ====================================================================
        viz_paths = None
        if generate_viz:
            print("\n🎨 STEP 4: Generating Visualizations...")
            print("\nOutput directory...", self.viz_path)
            self.visualizer = ModelVisualizer(
                self.evaluator, 
                self.feature_cols,
                self.viz_path
            )
            viz_paths = self.visualizer.generate_all_visualizations(
                feature_importance_df=feature_importance,
                save=save_viz,
                show=False
            )
        
        # ====================================================================
        # STEP 5: Log to MLflow
        # ====================================================================
        if log_to_mlflow:
            print("\n📝 STEP 5: Logging to MLflow...")
            self.mlflow_logger = MLflowLogger()
            
            with self.mlflow_logger.start_run(f"{self.algorithm}_pipeline"):
                # Log parameters
                params = {
                    'algorithm': self.algorithm,
                    'n_customers': n_customers,
                    'test_size': test_size,
                    **self.algo_params
                }
                self.mlflow_logger.log_params(params)
                
                # Log metrics
                self.mlflow_logger.log_metrics(metrics)
                
                # Log artifacts
                if viz_paths:
                    self.mlflow_logger.log_artifacts(viz_paths)
                
                # Log model
                self.mlflow_logger.log_model(
                    model, 
                    self.trainer.train_df,
                    predictions,
                    self.feature_cols,
                    registered_model_name=f"customer_churn_{self.algorithm}"
                )
                
                run_id = self.mlflow_logger.run_id
            
            self.mlflow_logger.end_run()
        
        # ====================================================================
        # STEP 6: Export Results
        # ====================================================================
        if export_results:
            print("\n💾 STEP 6: Exporting Results...")
            self.report_generator = ReportGenerator(self.evaluator, self.trainer)
            export_paths = self.report_generator.export_all(visualization_paths=viz_paths)
            
            # Generate GenAI prompt
            genai_prompt, prompt_path = self.report_generator.generate_genai_prompt()
            print("\n" + "="*80)
            print("📤 GenAI PROMPT FOR RESULTS EXPLANATION")
            print("="*80)
            print(genai_prompt)
        
        # ====================================================================
        # Pipeline Complete
        # ====================================================================
        print("\n" + "="*80)
        print("✅ PIPELINE COMPLETE!")
        print("="*80)
        
        if log_to_mlflow:
            print(f"MLflow Run ID: {run_id}")
            print(f"View in Databricks: Experiments → {self.algorithm}_pipeline")
        
        if export_results:
            print(f"\nExported files:")
            for format_name, filepath in export_paths.items():
                print(f"  - {format_name}: {filepath}")
        
        print("="*80)
        
        return {
            'model': model,
            'predictions': predictions,
            'metrics': metrics,
            'feature_importance': feature_importance,
            'visualizations': viz_paths,
            'exports': export_paths if export_results else None,
            'mlflow_run_id': run_id if log_to_mlflow else None
        }



📁 ml_pipeline/
├── 📄 data_generator.py       # Generate dummy data
├── 📄 model_trainer.py        # Train ML models (reusable for any algorithm)
├── 📄 model_evaluator.py      # Calculate metrics (algorithm-agnostic)
├── 📄 visualizer.py           # Generate charts (standalone)
├── 📄 mlflow_logger.py        # MLflow logging utilities
├── 📄 report_generator.py     # Export results to different formats
└── 📄 main_pipeline.py        # Orchestrate everything

![](./Folder_Structure.jpg)
